# Creating an airfoil based on the desired properties


This cell will analyze an airfoil defined by 3 variables (classic NACA airfoils) and the output is Cl

In [ ]:
import numpy as np
import sys
sys.path.append("../")

from src.model import load_model
from src.optimizer import evaluate

model = load_model("../models/best_model.pth")
norm_stats = np.load("../models/norm_stats.npz")


# NACA 2412  5° 
result = evaluate([0.02, 0.4, 0.12], alpha=5.0,
                  model=model, norm_stats=norm_stats)

print(f"Cl = {result['cl']:.3f}")

Cl = 0.839


Now that we have Cl from a NACA profile we can run a loop that will try to look for a profile that maches the requirements

In [27]:
def objective_cl(params, target_cl, alpha, model, norm_stats):
    """
    Returns ONE value (smaller = better).
    Here: quadratic error between predicted and target Cl.
    Cl = target_cl  -> error 0 (ideal).
    """
    aero = evaluate(params, alpha, model, norm_stats)
    err = (aero["cl"] - target_cl) ** 2
    return err

from scipy.optimize import differential_evolution

def design_for_cl(target_cl, alpha, model, norm_stats):
    """
    Finds NACA parameters [m, p, t] that yield the requested target Cl.
    """

    bounds = [
        (0.0,  0.08),   
        (0.1,  0.6),    
        (0.08, 0.20),   
    ]

    result = differential_evolution(
        objective_cl,
        bounds,
        args=(target_cl, alpha, model, norm_stats),  
        maxiter=15,      
        popsize=8,       
        seed=0,          
        polish=False,    
        tol=1e-4,
    )

    best = result.x
    aero = evaluate(best, alpha, model, norm_stats)
    return {
        "params": best,
        "cl": aero["cl"],
        "n_evals": result.nfev,   
    }

out = design_for_cl(target_cl=1.0, alpha=5.0,
                    model=model, norm_stats=norm_stats)

print(f"Target Cl:    1.000")
print(f"Found Cl:   {out['cl']:.3f}")
print(f"NACA parametry: m={out['params'][0]:.3f}, "
      f"p={out['params'][1]:.3f}, t={out['params'][2]:.3f}")
print(f"Number of evaluations: {out['n_evals']}")

Hledané Cl:    1.000
Nalezené Cl:   1.000
NACA parametry: m=0.050, p=0.458, t=0.097
Počet vyhodnocení: 384
